In [21]:
import os
import sys

import logging

%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


from src.onBench.utils.EDA import *


logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)


from src.onBench.entity.config_entity import DataIngestionConfig
from src.onBench.config.configuration import ConfigurationManager



In [22]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [23]:
"""config = ConfigurationManager()
data_ingestion_config = config.get_data_ingestion_config()
config=data_ingestion_config
"""

'config = ConfigurationManager()\ndata_ingestion_config = config.get_data_ingestion_config()\nconfig=data_ingestion_config\n'

In [24]:
"""database = config.database
schema = config.input_schema
input_table = config.candidate.input_table
input_table_pb = config.candidate.input_table_pb
columns_candidate = config.candidate.columns.candidate
columns_joborder = config.candidate.columns.joborder
days_nomission = config.candidate.days_nomission
"""

if(False):
    candidate_query = f"""
        CREATE OR REPLACE TABLE IT_DISCOVERY.ITA_EXP_ONBENCH.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED_ALL_EDA AS
        SELECT *
        FROM (
        WITH
        
        CTE_Cdc AS (
        SELECT o.dboffcode AS cdc,
        o.dboffname AS filiale,
        a.dbregname AS area,
        r.dbregname AS regione  
        FROM  it_discovery.consumer_int_model.PB_REFOFFICE_CLEANED o
        INNER JOIN  it_discovery.consumer_int_model.PB_REFREGION_CLEANED a  ON o.dboffregion = a.dbregcode  
        INNER JOIN  it_discovery.consumer_int_model.PB_REFREGION_CLEANED r  ON a.dbregfather = r.dbregcode
        ),
        CTE_Gruppo AS (
        SELECT g.dbclngroupid, 
        g.dbclngroupname, 
        c.canale,
        t.description AS canaledesc, 
        c.servrep, 
        c.isInternationalAccount
        FROM it_discovery.consumer_int_model.PB_FTCLNGROUP_CLEANED g (NOLOCK)
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCANALECOMM_CLEANED c (NOLOCK) ON c.parenttipo = 1 AND GETDATE() BETWEEN c.dtInizioVal AND c.dtFineVal AND c.parentID = g.dbclngroupid
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCOMBOS_CLEANED t ON t.type = 'CLN_KEY' AND t.itemdata = coalesce(c.canale,0)
        WHERE coalesce(c.isCorporateAccount,0) = 1
        ),
        candidate_onbench as(
        SELECT distinct
            cd.dbcandno AS candidate_id_pb,--CandidateId, 
        concat(coalesce(cd.dbcandnames, ''), ' ', coalesce(cd.dbcandsurname, '')) AS CandidateName,
        mp.cdc AS CdcIdAssTI,  
        mp.filiale AS CdcNameAssTI,
        mp.area AS AreaAssTI,
        mp.regione AS RegionAssTI,
        cd.dbAssuntoTI,
        cd.dbasstidataassunz,
            cd.dbasstinomission,
        cd.dbAssTIDataStab,
        cd.dbAssTIflgSgravioTri,
        cd.dbAssTIdtSgravioTriInizio,
        cd.dbAssTIdtSgravioTriFine,
        jo.dbjobordno AS JobOrderId,
        ra.dbAzNome AS CompanyName,
        cl.dbclnno AS ClientId,  
        cp.cmpname AS ClientName,  
        mj.cdc AS JobOrderCdcId,  
        mj.filiale AS JobOrderCdcName,  
        mj.area AS JobOrderArea,  
        mj.regione AS JobOrderRegion,  
        jo.dbjobdatecommence AS JobOrderStartDate,  
        jo.dbjobenddate AS JobOrderEndDate,  
        coldb.description AS JobOrderBusinessLine,  
        cojot.description AS JobOrderJobType, 
        copos.description AS JobOrderPosition,
        jo.dbiDtFinePrec AS JobOrderPrevEndDate,  
        coapp.description AS JobOrderApprendistato,  
        ex.dbiassignment AS JobOrderExtAssignment,  
        ex.dbicategory AS JobOrderExtCCNLCategory,
        cd.dbcanddob AS CandidateDOB, -- CM5803
        coalesce(grp.servrep, '') AS GroupCAM, -- CM5803
        CASE 
        WHEN (coalesce(cc.isKeyAccount, 0) <> 0) THEN cc.servRep
        ELSE '' 
        END AS CompanyKAM, -- CM5803,
        ccnlID,
        dbiccnldesc,
        cast(ex.dbinumeromensilita as int) as mensilita,
        REPLACE(ex.dbiclndeclgms,'.',',') as retr_mensile_aziendale,
        REPLACE(cast(ex.dbinumeromensilita as int)*ex.dbiclndeclgms, '.', ',') as RAL,
        vertical.DscVerticalAzienda,
        vertical.DscCanaleAzienda,
        dbAssTIDataFinePeriodoFormativo,
        CASE
        WHEN coalesce(mypath.Partecipante, 0) = 1 THEN 'SI'
        ELSE 'NO' END AS My_Path,
            delivery_cdc_jofilled.cdc as delivery_cdc_jofilled,
            delivery_cdc_jofilled.filiale as delivery_filiale_jofilled
        
        FROM it_discovery.consumer_int_model.PB_FTCANDIDATE_CLEANED cd
        INNER JOIN it_discovery.consumer_int_model.PB_FTJOBORDER_CLEANED jo ON jo.dbjobcandno = cd.dbcandno AND jo.dbjobenddate >= to_date(CURRENT_TIMESTAMP) 
            AND jo.dbjobdatecommence < DATEADD(DAY, 1, to_date(CURRENT_TIMESTAMP)) and jo.dbjobdatecommence >= coalesce(cd.dbAssTIDataStab,cd.dbasstidataassunz)
        INNER JOIN it_discovery.consumer_int_model.PB_FTJOBEXT_CLEANED ex on ex.dbjobordno = jo.dbjobordno
        INNER JOIN CTE_Cdc mp ON mp.cdc = cd.dbassticdc  
        INNER JOIN CTE_Cdc mj ON mj.cdc = jo.dbofficecode
        INNER JOIN it_discovery.consumer_int_model.PB_FTCLIENT_CLEANED cl ON cl.dbclnno = jo.dbjobclnno 
        INNER JOIN it_discovery.consumer_int_model.PB_FTCOMPANY_CLEANED cp ON cp.cmpno = cl.dbclnparent  
        INNER JOIN it_discovery.consumer_int_model.PB_REFAZIENDA_CLEANED ra ON ra.dbAzID = cd.dbAssuntoTI  
        LEFT JOIN CTE_Gruppo grp ON grp.dbclngroupid = cp.cmpParent
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCANALECOMM_CLEANED cc ON cc.parenttipo = 2 AND GETDATE() BETWEEN cc.dtInizioVal AND cc.dtFineVal AND cc.parentID = cp.cmpno
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCOMBOS_CLEANED coapp ON coapp.type = 'APPR' AND coapp.ItemData = coalesce(jo.Apprendistato, 0)
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCOMBOS_CLEANED coldb ON coldb.type = 'JLDB' AND coldb.itemdata = jo.lineaDiBusiness
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCOMBOS_CLEANED cojot ON cojot.type = 'JOTP' AND cojot.itemdata = jo.dbjobtype
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCOMBOS_CLEANED copos ON copos.type = 'JPOS' AND copos.itemdata = jo.dbjobpositiontype
        LEFT JOIN it_discovery.consumer_int_model.PB_FTCANDMYPATH_CLEANED mypath ON cd.dbcandno = mypath.dbcandno
        --LEFT JOIN MY_ITALY_NEW.dbo.v_ppccnlsalias ccnl (NOLOCK) ON ex.ccnlID=ccnl.chiave
        LEFT JOIN  it_discovery.consumer_int_model.PB_VWCANALIVERTICALCOMPANY_CLEANED vertical ON cp.cmpNo=vertical.CmpNo
            left JOIN it_discovery.consumer_int_model.PB_FTJOBORDER_CLEANED jfilled ON jfilled.dbjobcandno = cd.dbcandno and jfilled.dbjobstatus in (2,4) AND jfilled.dbjobenddate >= to_date(getdate()) AND jfilled.dbjobdatecommence <= to_date(getdate())
            left join CTE_Cdc as delivery_cdc_jofilled on jfilled.OfficeCodeCDR = delivery_cdc_jofilled.cdc
        
        WHERE cd.dbAssuntoTI IS NOT NULL
        AND coalesce(cd.dbAssTIDataCessaz,'2049-12-31') >= to_date(CURRENT_TIMESTAMP)
        ),
        --select * from candidate_onbench
        candidates AS (
                        SELECT
                            candidateid,
                            pb_id,
                            first_name, 
                            last_name, 
                            cap,
                            city,
                            provincia,
                            skill_set,
                            education_degree,
                            degree_list, 
                            salary_low, 
                            date_available, 
                            will_relocate, 
                            desired_location, 
                            parttime_preferenza_perc, 
                            turno_preferenza,
                            description
                        FROM IT_DISCOVERY.CONSUMER_INT_MODEL.CANDIDATE_CLEANED
                        WHERE isdeleted = 0
                        AND is_anonymized = 0
                    ),

                    ranked AS (
                        SELECT
                            dbjobordno,
                            dbjobcandno,
                            dbjobsite,
                            dbjobposition,
                            dbjobdatecommence,
                            dbjobenddate,
                            dbiparttimepcnt,
                            ROW_NUMBER() OVER (
                                PARTITION BY dbjobcandno
                                ORDER BY dbjobdatecommence
                            ) AS rn_first,
                            ROW_NUMBER() OVER (
                                PARTITION BY dbjobcandno
                                ORDER BY dbjobdatecommence DESC
                            ) AS rn_last
                        FROM IT_DISCOVERY.CONSUMER_INT_MODEL.pb_ftjoborder_cleaned
                    ),

                    jobs_ranked AS (
                        SELECT
                            dbjobordno,
                            dbjobcandno,
                            dbjobsite,
                            dbjobposition,
                            dbjobdatecommence,
                            dbjobenddate,
                            dbiparttimepcnt,
                            CASE
                                WHEN rn_first = 1 THEN 'first'
                                WHEN rn_last = 1 THEN 'last'
                                ELSE 'intermediate'
                            END AS job_rank
                        FROM ranked
                    ),

                    jobs_ranked_info AS (
                        SELECT
                            jo.dbjobordno,
                            jo.dbjobcandno,
                            jo.job_rank,
                            jo.dbjobsite,
                            jo.dbjobposition,
                            jo.dbjobdatecommence,
                            jo.dbjobenddate,
                            jo.dbiparttimepcnt,
                            je.dbilevel,
                            je.dbiqualification,
                            ccnl_codice,
                            ccnl_descrizione
                        FROM jobs_ranked jo
                        INNER JOIN IT_DISCOVERY.CONSUMER_INT_MODEL.pb_ftjobext_cleaned je
                            ON jo.dbjobordno = je.dbjobordno
                        LEFT JOIN IT_DISCOVERY.CONSUMER_INT_MODEL.v_ppccnlsalias cc
                            ON cc.chiave = je.ccnlid
                    ),

                    jobs_ranked_agg AS (
                        SELECT
                            dbjobcandno,

                            MAX(CASE WHEN job_rank = 'first' THEN dbjobsite END) AS job_site,
                            MAX(CASE WHEN job_rank = 'first' THEN dbjobposition END) AS job_first,

                            NULLIF(
                                LISTAGG(
                                    CASE WHEN job_rank = 'intermediate' THEN dbjobposition END,
                                    ', '
                                ) WITHIN GROUP (ORDER BY dbjobdatecommence),
                                ''
                            ) AS job_intermediate,

                            MAX(CASE WHEN job_rank = 'last' THEN dbjobposition END) AS job_last,

                            MIN(dbjobdatecommence) AS start_date,
                            MAX(dbjobenddate) AS end_date,

                            MAX(CASE WHEN job_rank = 'first' THEN dbiparttimepcnt END) AS parttime_perc,
                            MAX(CASE WHEN job_rank = 'first' THEN dbilevel END) AS ccnl_level,
                            MAX(CASE WHEN job_rank = 'first' THEN dbiqualification END) AS legal_category,
                            MAX(CASE WHEN job_rank = 'first' THEN ccnl_codice END) AS ccnl_code,
                            MAX(CASE WHEN job_rank = 'first' THEN ccnl_descrizione END) AS ccnl_description

                        FROM jobs_ranked_info
                        GROUP BY dbjobcandno
                    )

                    SELECT
                        cc.*,
                        IFF(
                            COALESCE(
                                NULLIF(TRIM(description), ''),
                                NULLIF(TRIM(skill_set), ''),
                                NULLIF(TRIM(education_degree), ''),
                                NULLIF(TRIM(degree_list), '')
                            ) IS NULL,
                            NULL,
                            CONCAT(
                                'Curriculum: ', COALESCE(description, ''), ' | ',
                                'Skills: ', COALESCE(skill_set, ''), ' | ',
                                'Titoli di studio richiesti: ', COALESCE(education_degree, ''), ' | ',
                                'Certificazioni: ', COALESCE(degree_list, ''), ' | '
                            )
                        ) AS description_all,
                        {", ".join(f"jri.{source} AS {alias}" for source, alias in columns_joborder.items())}, --jri.*,--
                        --cob.status_candidato,
                    FROM candidates cc
                    INNER JOIN candidate_onbench cob
                        ON cc.pb_id = cob.candidate_id_pb
                    LEFT JOIN jobs_ranked_agg jri
                        ON cc.pb_id = jri.dbjobcandno
        )
        """
    #session.sql(candidate_query)


In [25]:
df = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED--_ALL_EDA
                     --where status = 'No mission'
                 """).to_pandas()

"""
df = df.rename(columns={"SKILL_SET": "SKILLS"})
df = df.rename(columns={"EDUCATION_DEGREE": "EDUCATION"})
df = df.rename(columns={"DEGREE_LIST": "CERTIFICATIONS"})
df = df.rename(columns={"SALARY_LOW": "SALARY"})
df = df.rename(columns={"DESCRIPTION": "CURRICULUM"})
df = df.rename(columns={"DESCRIPTION_ALL": "DESCRIPTION"})
df = df.rename(columns={"PROVINCIA": "PROVINCE"})
"""


'\ndf = df.rename(columns={"SKILL_SET": "SKILLS"})\ndf = df.rename(columns={"EDUCATION_DEGREE": "EDUCATION"})\ndf = df.rename(columns={"DEGREE_LIST": "CERTIFICATIONS"})\ndf = df.rename(columns={"SALARY_LOW": "SALARY"})\ndf = df.rename(columns={"DESCRIPTION": "CURRICULUM"})\ndf = df.rename(columns={"DESCRIPTION_ALL": "DESCRIPTION"})\ndf = df.rename(columns={"PROVINCIA": "PROVINCE"})\n'

In [26]:
df["CCNL_CODE"] = df["CCNL_CODE"].astype("Int64").astype("string")

df[df.select_dtypes(include=["object", "string"]).columns] = (
    df.select_dtypes(include=["object", "string"]).apply(lambda col: col.str.strip())
)


In [27]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "esperienza", "lavoro", "occupazione", "posizione", "ruolo", "azienda",
    "experience", "job", "position"

    # ruoli
    "addetto", "operaio", "impiegato", "responsabile", "cassiere", "magazziniere",
    "analista", "sviluppatore", "developer", "autista", "segretaria", "tecnico", "consulente",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diploma", "laurea", "master", "istituto", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "inglese", "informatica", "patente",
    "english", "computer", "driving license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "srl", "spa", "cooperativa", "comune", "regione"
    "regione", "municipality", "company", "organization", "corporation"
]

df["IS_VALID_CV"] = df["CURRICULUM"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_cv = np.sum(df["IS_VALID_CV"])/df.shape[0]

print(f"""Percentage of invalid CVs (custom keywords matching): {1-is_valid_cv:.2%}""")




Percentage of invalid CVs (custom keywords matching): 47.32%


# MISSING CVs #

Other than using CV length to identify real CVs (50 characters), I also define a set of keywords that are normally found in a CV and I check whether a CV contains any. ù
So, if the CV contains at least 50 characters and at least one of the keywords mentioned, I classify it as real.
According to this method, 47% of candidate CS are not valid (are emtpy or contain meaningless text)

In [28]:
general_info(df)


Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   CANDIDATE_ID_BH              429 non-null    int32         
 1   SALARY                       4 non-null      float64       
 2   DATE_AVAILABLE               366 non-null    datetime64[ns]
 3   WILL_RELOCATE                429 non-null    bool          
 4   START_DATE                   429 non-null    datetime64[ns]
 5   END_DATE                     429 non-null    datetime64[ns]
 6   DB_ASS_TI_NOMISSION          429 non-null    datetime64[ns]
 7   LAST_ONBENCH_JOB_START_DATE  100 non-null    datetime64[ns]
 8   LAST_ONBENCH_JOB_END_DATE    100 non-null    datetime64[ns]
 9   DAYS_SINCE_ONBENCH           429 non-null    int16         
 10  DAYS_SINCE_LAST_JOB          429 non-null    int32         
 11  DAYS_ONBENCH_LAST            429 non-n

# COMMENTS on missing/unique values #

9085 candidates

Comments on candidate missing data (Bullhorn):
-   salary (99% missing)
-   turno_preferenza (99.5% missing)
-   parttime_preferenza_perc (99.7% missing)
-   skills (82% missing)
-   education (66% missing)
-   certifications (100% missing)
-   curriculum 48% missing
-   description (34% missing. this means that curriculum, skills, education and certifiations are all simultaneaously missing)

Comments on candidate missing data (PowerBase):
-   parttime_perc (first job) (88% missing)
-   CCNL level (first job) (0% missing)
-   CCNL code (first job) (1.4%)
-   CCNL description (first job) (1.4%)
-   legal_category (first job) (0% missing)


Some comments on unique values:
- 11 unique values for education
- 776 unique first jobs
- 172 unique CCNL codes
- 209 unique CCNL descriptions
- 6 unique legal categories

In [29]:
columns_to_check = ["EDUCATION", "CERTIFICATIONS"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)


Unique values in 'EDUCATION':
['diploma' 'laurea' 'laurea breve' 'non specificato' 'scuola media'
 'scuola secondaria' 'universita' None]
----------------------------------------
Unique values in 'CERTIFICATIONS':
[None]
----------------------------------------


In [30]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)



Unique values in 'CCNL_LEVEL':
['1' '2' '2tr' '3' '3 ap liv. 1' '4' '4 junior' '4 senior'
 '4 super (ex 5)' '4/1' '5' '5 fascia 1' '6' '6 junior' '6 super (ex 7)'
 '7' '8' 'a' 'a1' 'ae1' 'ae2' 'ap prima' 'area 3 liv. 1' 'area3liv.1' 'b'
 'b1' 'b2' 'bs super' 'c' 'c/1' 'c/2' 'c/3' 'c1' 'c2' 'c3' 'd' 'd/2' 'd1'
 'd2' 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'f' 'g' 'g1' 'h' 'i' 'iii'
 'op centrato' 'op consolidato' 'prof.s. e funz.' 'v']
----------------------------------------
Unique values in 'CCNL_CODE':
<StringArray>
['1020',  '106', '1270',  '130', '1380', '1390', '1620', '1790', '1810',
 '1820', '1870', '1920', '1930', '1980', '2070',  '220', '2220',  '230',
 '2310', '2360',  '240', '2470',  '250', '2630',  '330',   '38',   '39',
  '530', '5750',  '710',  '720',   '88',   '89', '9013', '9019', '9050',
 '9275', '9332', '9335', '9354', '9357', '9408', '9504', '9633', '9646',
 '9662', '9669', '9674', '9683', '9694', '9705', '9719', '9726', '9746',
 '9776', '9791', '9796', '9808', '9816', '9817', 

In [31]:
columns_to_check = ["JOB_FIRST"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)

Unique values in 'JOB_FIRST':
['add. rifinitura- qualità e confez. pelletteria'
 'add. rifinitura/ qualità e confez. pelletteria' 'addetti global custody'
 'addetto agli allestimenti interni ferroviari'
 'addetto ai controlli contabili' 'addetto al magazzino'
 'addetto al magazzino settore abbigliamento' 'addetto al montaggio'
 'addetto alimentazione mulini' 'addetto alla composizione'
 'addetto alla fatturazione' 'addetto alla laminazione di fibra di vetro'
 'addetto alla logistica' 'addetto alla pescheria'
 'addetto alla pianificazione acquisti'
 'addetto alla pressa per lo stampaggio'
 'addetto alla produzione di pasta alimentare'
 'addetto alla produzione di semiconduttori' 'addetto alla reception'
 'addetto alla vendita attrezzature audio video' 'addetto alle pulizie'
 'addetto alle pulizie negli stabili' 'addetto allo sportello bancario'
 'addetto all’assemblaggio di macchinari industriali'
 'addetto all’estrusione per materie plastiche'
 'addetto assemblaggio strumenti di precis

In [32]:
b1 = bar_plot(df, 'EDUCATION', rotation=25)
b2 = bar_plot(df, 'CERTIFICATIONS', rotation=25)
b3 = bar_plot(df, 'IS_VALID_CV', rotation=25)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [EDUCATION]   (COUNT)
   .Bars.II  :Bars   [CERTIFICATIONS]   (COUNT)
   .Bars.III :Bars   [IS_VALID_CV]   (COUNT)

In [33]:
b1 = bar_plot(df, 'CAP', rotation=25)
b2 = bar_plot(df, 'CITY', rotation=25)
b3 = bar_plot(df, 'PROVINCE', rotation=25)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [CAP]   (COUNT)
   .Bars.II  :Bars   [CITY]   (COUNT)
   .Bars.III :Bars   [PROVINCE]   (COUNT)

In [34]:
b1 = bar_plot(df, 'CCNL_LEVEL')#, label_range=30)
b2 = bar_plot(df, 'CCNL_CODE')#, label_range=30)
b3 = bar_plot(df, 'LEGAL_CATEGORY', rotation=25)#, label_range=30)
b4 = bar_plot(df, 'CCNL_DESCRIPTION', rotation=25, height=600, width=1200)

(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [CCNL_LEVEL]   (COUNT)
   .Bars.II  :Bars   [CCNL_CODE]   (COUNT)
   .Bars.III :Bars   [LEGAL_CATEGORY]   (COUNT)
   .Bars.IV  :Bars   [CCNL_DESCRIPTION]   (COUNT)

In [35]:
b1 = bar_plot(df, 'JOB_FIRST', rotation=25, height=600, width=1200)#, label_range=30)
b2 = bar_plot(df, 'JOB_LAST', rotation=25, height=600, width=1200)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [JOB_FIRST]   (COUNT)
   .Bars.II :Bars   [JOB_LAST]   (COUNT)

In [36]:
b1 = bar_plot(df, 'JOB_SITE', rotation=25, height=800, width=2400)#, label_range=30)
b1

:Bars   [JOB_SITE]   (COUNT)

In [43]:
h1 = hist_plot(df, "PARTTIME_PERC", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [PARTTIME_PERC]   (Frequency)

# VACANCY #

In [44]:
"""database = config.database
schema = config.schema
input_table_vacancy = config.vacancy.input_table
columns = config.vacancy.columns
days_prior = config.vacancy.days_prior
"""
if(False):
    vacancy_query = f"""
    CREATE OR REPLACE TABLE IT_DISCOVERY.ITA_EXP_ONBENCH.MPG_IT_ONBENCH_TEST_JOBORDER_CLEANED_ALL_EDA AS
    WITH 
    ppccnlsalias AS(
    SELECT 
        chiave,
        LISTAGG(DISTINCT ccnl_codice, '||') WITHIN GROUP (ORDER BY ccnl_codice) AS ccnl_codice,
        LISTAGG(DISTINCT ccnl_descrizione, '||') WITHIN GROUP (ORDER BY ccnl_descrizione) AS ccnl_descrizione
    FROM IT_DISCOVERY.CONSUMER_INT_MODEL.v_ppccnlsalias
    GROUP BY chiave
    ),
    joborder_pb AS (
    SELECT
        jo.dbjobordno,
        jo.dbjobcandno,
        jo.dbjobsite as job_site,
        jo.dbjobposition as job_position,
        jo.dbjobdatecommence as start_date,
        jo.dbjobenddate as end_date,
        jo.dbiparttimepcnt as parttime_perc,
        je.dbilevel as ccnl_level,
        je.dbiqualification as legal_category,
        ccnl_codice as ccnl_code,
        ccnl_descrizione as ccnl_description
    FROM IT_DISCOVERY.CONSUMER_INT_MODEL.pb_ftjoborder_cleaned jo
    INNER JOIN IT_DISCOVERY.CONSUMER_INT_MODEL.pb_ftjobext_cleaned je
        ON jo.dbjobordno = je.dbjobordno
    LEFT JOIN ppccnlsalias cc
        ON cc.chiave = je.ccnlid
    ),
    filtered_jc AS (
        SELECT 
        joborderid,
        pb_id,
        jobtitle,
        citta,
        regione,
        salary,
        data_inizio_validita,
        skill_list,
        part_time_percent,
        titoli_richiesti,
        descrizione_uso_interno,
        testo_pubblicazione,
        richieste_aggiuntive,
        requisiti,
        isopen, 
        status, 
        datelastmodified,
        FROM IT_DISCOVERY.CONSUMER_INT_MODEL.JOBORDER_CLEANED
        WHERE isopen = 1 AND status = 'Accepting Candidates'
    ),
    jsc_agg AS (
        SELECT
            joborderid,
            MAX(data_ultimo_cambio_status) AS last_status_change
        FROM IT_DISCOVERY.CONSUMER_INT_MODEL.JOBSUBMISSION_CLEANED
        GROUP BY joborderid
    ),
    joborder_bh AS (
        SELECT filtered_jc.*, jsc_agg.last_status_change
        FROM filtered_jc LEFT JOIN jsc_agg ON filtered_jc.joborderid = jsc_agg.joborderid

    ),
    joborder AS (
    select  *
    from joborder_bh jb left join joborder_pb jp on cast(jb.pb_id  as string) = cast(jp.dbjobordno as string)
    )
    SELECT
        *, 
        CASE 
            WHEN datelastmodified IS NULL
            AND last_status_change IS NULL
                THEN NULL
            ELSE GREATEST(
                    COALESCE(datelastmodified, TO_TIMESTAMP('0001-01-01 00:00:00')),
                    COALESCE(last_status_change, TO_TIMESTAMP('0001-01-01 00:00:00'))
                )
        END AS date_last_active,
        IFF(
            COALESCE(
                NULLIF(TRIM(skill_list), ''),
                NULLIF(TRIM(part_time_percent), ''),
                NULLIF(TRIM(titoli_richiesti), ''),
                NULLIF(TRIM(DESCRIZIONE_USO_INTERNO), ''),
                NULLIF(TRIM(TESTO_PUBBLICAZIONE), ''),
                NULLIF(TRIM(RICHIESTE_AGGIUNTIVE), ''),
                NULLIF(TRIM(REQUISITI), '')
            ) IS NULL,
            NULL,
            CONCAT(
                'Skills richieste: ', COALESCE(skill_list, ''), ' | ',
                --'Percentuale part time: ', COALESCE(part_time_percent, ''), ' | ',
                'Titoli di studio richiesti: ', COALESCE(titoli_richiesti, ''), ' | ',
                'DESCRIZIONE USO INTERNO: ', COALESCE(DESCRIZIONE_USO_INTERNO, ''), ' | ',
                'TESTO PUBBLICAZIONE: ', COALESCE(TESTO_PUBBLICAZIONE, ''), ' | ',
                'RICHIESTE AGGIUNTIVE: ', COALESCE(RICHIESTE_AGGIUNTIVE, ''), ' | ',
                'REQUISITI: ', COALESCE(REQUISITI, '')
            )
        ) AS description
    FROM joborder

    """
    session.sql(vacancy_query).collect()

In [45]:
df_vacancy = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_JOBORDER_CLEANED
                 """).to_pandas()

In [46]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "ricerc", "cerc", "esperienza", "lavor", "occupazion", "posizion", "ruol", "aziend",
    "experience", "job", "position"

    # ruoli
    "addett", "operai", "impiegat", "responsabil", "cassier", "magazzinier",
    "analist", "sviluppator", "developer", "autist", "segretar", "tecnic", "consulent",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diplom", "laure", "master", "istitut", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "ingles", "informatic", "patent",
    "english", "computer", "driving license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "aziend", "ente", "enti", "srl", "spa", "cooperativ", "comun", "region"
    "regione", "municipality", "company", "organization", "corporation"
]

df_vacancy["IS_VALID_VACANCY"] = df_vacancy["DESCRIPTION"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_vacancy = np.sum(df_vacancy["IS_VALID_VACANCY"])/df_vacancy.shape[0]

print(f"""Percentage of invalid vacancies (custom keywords matching): {1-is_valid_vacancy:.2%}""")




Percentage of invalid vacancies (custom keywords matching): 39.76%


In [47]:
general_info(df_vacancy)

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31799 entries, 0 to 31798
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   JOBORDERID               31799 non-null  int32         
 1   SALARY                   31799 non-null  float64       
 2   DATE_AVAILABLE           21656 non-null  datetime64[ns]
 3   DATE_LAST_ACTIVE         31799 non-null  datetime64[ns]
 4   JOB_TITLE                31799 non-null  object        
 5   LOCATION                 27641 non-null  object        
 6   REGION                   26531 non-null  object        
 7   SKILLS                   641 non-null    object        
 8   PARTTIME_PERC            2126 non-null   object        
 9   TITOLI_RICHIESTI         270 non-null    object        
 10  RICHIESTE_AGGIUNTIVE     437 non-null    object        
 11  REQUISITI                16848 non-null  object        
 12  JOB_SITE                 

# COMMENTS on missing/unique values #

32024 vacancies

Comments on candidate missing data (Bullhorn):
-   jobtitle 0%
-   salary 0% 
-   citta 13%
-   regione 16%
-   skills 98%
-   parttime_perc 94%
-   testo_pubblicatione 30%
-   requisiti 47%
-   titoli_richiesti 99%
-   descrizione_uso_interno 30%
-   richieste_aggiuntive 99%
-   description (0.03% missing. this means that skills, testo_pubblicazione, requisiti, titoli_richiesti, descrizione_us_interno are simultaneously missing)

Comments on candidate missing data (PowerBase):
-   job_site 92%
-   job_position 92%
-   parttime_perc 97% missing
-   CCNL level 92% missing
-   CCNL code 92%
-   CCNL description 92%
-   legal_category 92%


Some comments on unique values:
- 12 unique values for titoli_richiesti
- 400 unique regions
- 162 unique CCNL codes
- 196 unique CCNL descriptions
- 103 unique CCNL levels
- 6 unique legal categories

In [48]:
columns_to_check = ["TITOLI_RICHIESTI", "TESTO_PUBBLICAZIONE", "REQUISITI", "RICHIESTE_AGGIUNTIVE", "DESCRIZIONE_USO_INTERNO"]

unique_limit = 100
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")



13 unique values in TITOLI_RICHIESTI over 270 total elements:

['diploma' 'diploma,laurea breve' 'diploma,laurea breve,laurea'
 'diploma,non specificato' 'laurea' 'laurea breve' 'laurea breve,laurea'
 'laurea breve,laurea,diploma' 'laurea breve,laurea,master'
 'laurea,laurea breve,master' 'non specificato' 'non specificato,diploma'
 None]
----------------------------------------

19711 unique values in TESTO_PUBBLICAZIONE over 22028 total elements:
(truncated to first 100)
['\n\n\n\n\n\n\n\nexperis è la società it & technology del gruppo manpower.\nle persone e le loro competenze sono il nostro successo e insieme ci poniamo come partner tecnologici dei nostri clienti attraverso servizi e soluzioni.\nvogliamo essere artefici, proprio insieme a te, di un ambiente di lavoro innovativo, stimolante e flessibile, attraverso i nostri valori: people, knowledge e innovation.\nexperis ricerca un\xa0obsolescence engineer\xa0per\xa0potenziare la divisione\xa0aerospazio&difesa\xa0sulla sede di:\xa0

In [49]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

unique_limit = 100
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")



106 unique values in CCNL_LEVEL over 2637 total elements:
(truncated to first 100)
['1' '1 super q' '140' '2' '2 ap liv. 2' '3' '3 amministr.' '3 ap liv. 1'
 '3 super' '3/1' '3/2' '3/3' '3a' '4' '4 junior' '4 senior' '4 super'
 '4 super (ex 5)' '4/1' '4/3' '5' '5 (ex 6)' '5/0' '5/1' '5a' '6'
 '6 junior' '6 senior' '6 super' '6 super (ex 7)' '6i' '7' '8' 'a' 'a1'
 'a2' 'a3' 'ae1' 'ae2' 'ap prima' 'ap seconda' 'ap terza' 'area 2'
 'area 3' 'area 3 liv. 1' 'area assistenti' 'area esecutiva'
 'area istruttori' 'area operatori' 'area supporto' 'b' 'b classe 1'
 'b super' 'b/2' 'b/3' 'b1' 'b1 super' 'b2' 'b2.' 'b3' 'b3.' 'b4'
 'bs super' 'c' 'c super' 'c/1' 'c/2' 'c/3' 'c0' 'c1' 'c2' 'c2 (ex d1)'
 'c3' 'd' 'd/1' 'd/2' 'd1' 'd1 (ex 6)' 'd1 (ex d2)' 'd1s' 'd2' 'd2 (ex e)'
 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'eq e funzionari' 'ex 1-2 area' 'f'
 'f 1 posiz.' 'f2' 'g' 'g1' 'h' 'i' 'junior' 'op base' 'op centrato']
----------------------------------------

161 unique values in CCNL_CODE over 2636 total 

In [50]:
b1 = bar_plot(df_vacancy, 'LOCATION', rotation=25)#, label_range=30)
b2 = bar_plot(df_vacancy, 'REGION', rotation=25)#, label_range=30)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [LOCATION]   (COUNT)
   .Bars.II :Bars   [REGION]   (COUNT)

In [51]:
h1 = hist_plot(df, "SALARY", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [SALARY]   (Frequency)